# ROUT Outlier Detection

Outliers in dose-response data distort fitted parameters, especially EC50. ROUT (Robust regression and Outlier Removal) detects outliers by fitting with a Lorentzian merit function (which down-weights extremes), computing a Robust SD of Residuals, and applying Benjamini-Hochberg FDR control. This is the first Python implementation of the algorithm.

**Reference:** Motulsky & Brown, BMC Bioinformatics 2006, 7:123.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from openfit import Fit, rout_outliers

## Generate clean data then inject outliers

Simulate a Hill4P curve, then corrupt two points to act as outliers.

In [ ]:
rng = np.random.default_rng(7)
x = np.array([0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0])
# True 4PL
def hill4p(x, B, T, E, H):
    return B + (T - B) / (1 + (E / x) ** H)
y_true = hill4p(x, 2.0, 96.0, 8.0, 1.3)
y = y_true * (1 + rng.normal(0, 0.10, size=len(x)))
y = np.clip(y, 0.1, None)

# Inject two outliers: point at x=1.0 and x=30.0
y_outlier = y.copy()
y_outlier[2] = y[2] * 4.5   # x=1.0: spike up
y_outlier[5] = y[5] * 0.15  # x=30.0: spike down

print("Clean y:   ", np.round(y, 2))
print("Outlier y: ", np.round(y_outlier, 2))
print(f"Outlier indices: 2 (x={x[2]}) and 5 (x={x[5]})")

## Fit without outlier removal (naive)

Show how outliers distort the EC50 estimate.

In [ ]:
naive_fit = Fit("hill4p", x, y_outlier, weights="1/y2").run()
true_ec50 = 8.0
print(f"True EC50:  {true_ec50:.2f}")
print(f"Naive EC50: {naive_fit.params['EC50']:.2f}  (error: {abs(naive_fit.params['EC50']-true_ec50)/true_ec50*100:.1f}%)")
print(f"Naive R\u00b2:   {naive_fit.r_squared:.4f}")

## Run ROUT (Q=0.01)

Q is the maximum acceptable FDR (fraction of true non-outliers incorrectly flagged). Q=0.01 is the recommended default.

In [ ]:
rout = rout_outliers("hill4p", x, y_outlier, weights="1/y2", Q=0.01)
print(f"ROUT detected {rout.n_outliers} outlier(s)")
print(f"Outlier mask:    {rout.outlier_mask}")
print(f"Outlier indices: {rout.outlier_indices}")
print(f"Correctly identified: {set(rout.outlier_indices) == {2, 5}}")

## Refit without outliers

In [ ]:
x_clean = x[~rout.outlier_mask]
y_clean = y_outlier[~rout.outlier_mask]
clean_fit = Fit("hill4p", x_clean, y_clean, weights="1/y2").run()

print(f"True EC50:   {true_ec50:.2f}")
print(f"Naive EC50:  {naive_fit.params['EC50']:.2f}  (error: {abs(naive_fit.params['EC50']-true_ec50)/true_ec50*100:.1f}%)")
print(f"Clean EC50:  {clean_fit.params['EC50']:.2f}  (error: {abs(clean_fit.params['EC50']-true_ec50)/true_ec50*100:.1f}%)")
print(f"\nNaive R\u00b2:  {naive_fit.r_squared:.4f}")
print(f"Clean R\u00b2:  {clean_fit.r_squared:.4f}")

## Visualise: before and after

In [ ]:
x_fine = np.logspace(np.log10(x.min()), np.log10(x.max()), 300)

def eval_hill4p(x_arr, params):
    p = params
    return p["Bottom"] + (p["Top"] - p["Bottom"]) / (1 + (p["EC50"] / x_arr) ** p["HillSlope"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, fit, title, show_outliers in [
    (axes[0], naive_fit, "Naive fit (with outliers)", True),
    (axes[1], clean_fit, "Clean fit (ROUT removed)", False),
]:
    y_curve = eval_hill4p(x_fine, fit.params)
    ax.semilogx(x_fine, y_curve, "b-", linewidth=2, label="Fit")
    
    # Plot clean points
    mask = rout.outlier_mask
    ax.semilogx(x[~mask], y_outlier[~mask], "ko", markersize=7, label="Data")
    if show_outliers:
        ax.semilogx(x[mask], y_outlier[mask], "r*", markersize=12, label="Outliers", zorder=5)
    
    ax.axvline(fit.params["EC50"], color="blue", linestyle="--", alpha=0.6,
               label=f"EC50={fit.params['EC50']:.2f}")
    ax.axvline(true_ec50, color="green", linestyle=":", alpha=0.8, label=f"True EC50={true_ec50}")
    ax.set_xlabel("Concentration")
    ax.set_ylabel("Response")
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/rout_outliers.png", dpi=100, bbox_inches="tight")
plt.show()

## Effect of Q parameter

Lower Q = stricter = fewer flags. Try Q=0.001 and Q=0.1 to see the sensitivity of the FDR threshold.

In [ ]:
for Q in [0.001, 0.01, 0.1]:
    r = rout_outliers("hill4p", x, y_outlier, weights="1/y2", Q=Q)
    print(f"Q={Q:.3f}: {r.n_outliers} outlier(s) detected — indices {list(r.outlier_indices)}")

## Summary

- ROUT correctly identified both injected outliers without removing any clean points.
- EC50 error dropped significantly (naive) to near-zero (clean) after ROUT.
- Always visually inspect flagged points before accepting removal.
- Q=0.01 is appropriate for most dose-response applications.
- **Reference:** Motulsky & Brown, BMC Bioinformatics 2006, doi:10.1186/1471-2105-7-123.